# Qwen3.6 迁移到 vime 详解

## 写给 Megatron / MindSpeed / vime 新手

这份 Notebook 会把 Qwen3.6-35B-A3B 迁移到 vime 训练框架的**每一处改动**，放在完整框架上下文中讲解。

**你不会只看到 diff — 你会看到改动前框架长什么样、改动后变成了什么、以及为什么。**

### 阅读指南

按章节顺序阅读：每一章建立在前一章的概念之上。

1. **术语速查** — 先了解关键词
2. **框架全景** — 理解 vime/mbridge/Megatron 三者的关系和角色，特别是两种权重同步模式
3. **Qwen3.6 模型结构** — 理解要迁移的模型长什么样
4. **Bridge 继承链** — 理解权重转换框架的运转机制
5. **迁移方案：Route B** — 为什么选融合 in_proj，不选保持 4 路独立
6. **逐文件详解** — 每个文件的改动 + 原因 + 与框架的衔接（11 个文件）
7. **完整数据流** — 从 HF checkpoint 到 RL 训练的端到端流程（含 raw 模式路径）
8. **总结** — 硬约定 + 完整文件清单

## 1. 关键术语速查

| 缩写 | 全称 | 通俗解释 |
|------|------|---------|
| **GDN** | Gated DeltaNet | 一种线性注意力机制，替代标准 Self-Attention。核心是门控递归：$S_t = g_t \cdot S_{t-1} + v_t k_t^\top$ |
| **TP** | Tensor Parallel | 把一个大矩阵乘法沿某一维切开，分到多个 GPU 并行算 |
| **SP** | Sequence Parallel | 把 LayerNorm / Dropout 沿序列维切分，配合 TP 省显存 |
| **CP** | Context Parallel | 把一条很长的序列切成多段分到多个 GPU。核心是 Ulysses all-to-all |
| **EP** | Expert Parallel | MoE 中专家的并行：每个 GPU 只存 1/EP 个专家 |
| **raw 模式** | `--megatron-to-hf-mode raw` | 训练中直接从 Megatron 参数转 HF 格式，不走 Bridge 完整管道 |
| **mbridge** | Megatron-Bridge | HF 权重 ↔ Megatron 权重的转换桥梁（一个 Python 库） |
| **fused in_proj** | 融合投影层 | 把 4 个 `nn.Linear` (qkv, z, b, a) 合并成 1 个 `ColumnParallelLinear` |
| **dist-ckpt** | Distributed Checkpoint | Megatron 的 checkpoint 格式：按 TP/PP/DP 分片存储，加载时不关心并行配置 |
| **layerwise reload** | 逐层重载 | vLLM 的权重热更新机制：临时撤销 post-load 融合，使参数可被 `weight_loader` 重新加载 |
| **sharded_state_dict** | 分片状态字典 | 声明每参数在 TP/EP 维度上的切分方式，保证 dist-ckpt 跨并行度 resharding |

## 2. 框架全景：谁是谁？

### 2.1 vime / mbridge / Megatron 三者的关系

```
vime (训练框架)
├── vime_plugins/              ← 模型 + bridge 插件
│   ├── models/
│   │   ├── qwen3_5.py         ← GDN forward + TP/CP/SP
│   │   ├── hf_attention.py    ← HF 风格 Attention wrapper
│   │   ├── qwen_gdn_backend.py← Kernel dispatch (fla/flashqla/npu)
│   │   └── gdn_cp_utils.py   ← CP 工具函数
│   └── mbridge/
│       ├── qwen3_5.py         ← Bridge: 权重映射 + 融合/拆分
│       └── gdn_param_mapping.py← 纯 torch GDN 权重转换函数
├── vime/backends/
│   ├── megatron_utils/
│   │   ├── megatron_to_hf/
│   │   │   └── qwen3_5.py     ← raw 模式 MCore→HF 转换器
│   │   └── update_weight/     ← 权重同步 (tensor IPC / distributed)
│   └── vllm_utils/
│       └── vllm_engine.py     ← vLLM 引擎管理 + sleep mode
└── vime/utils/
    └── arguments.py           ← CLI 参数注册

外部依赖:
├── mbridge (库)               ← Bridge/LLMBridge/Qwen2MoEBridge
└── Megatron-LM                ← GPTModel/ColumnParallelLinear/TP/CP/PP
```

vime_plugins 调用 mbridge 做权重转换，调用 Megatron 做分布式训练。

### 2.2 vime 的 RL 训练循环（宏观）

```python
for step in range(num_steps):
    responses = rollout_engine.generate(prompts)           # Step 1: Rollout
    rewards = reward_model(responses)                      # Step 2: Reward
    loss = actor.megatron_forward(prompts, responses, rewards)
    loss.backward(); optimizer.step()                      # Step 3: 训练
    update_weights_from_tensor(actor_model, rollout_engine)# Step 4: 权重同步
```

### 2.3 mbridge 是如何转换权重的？

mbridge 的核心是 `Bridge` 基类（`mbridge/core/bridge.py`）。`load_weights` 方法（HF→MCore）的执行流程：

| 步骤 | 操作 | 说明 |
|------|------|------|
| (1) | 读入 HF safetensors | `in_proj_qkv [qkv_dim, h]` + `in_proj_z [v_dim, h]` + `in_proj_b [nv, h]` + `in_proj_a [nv, h]` + conv1d ... |
| (2) | `Bridge._weight_to_mcore_format()` | 每条 Megatron 参数名 → 对应 HF 张量的**融合**逻辑（如 4 个独立 weight → 1 个 fused in_proj） |
| (3) | 融合后 | `fused in_proj [in_proj_dim, h]` |
| (4) | `Bridge._weight_split_across_tp()` | 沿 TP 维切成 `tp_size` 份分发给各 rank |

每个 Bridge 子类需要重写：

| 方法 | 方向 | 输入 | 输出 |
|------|------|------|------|
| `_weight_name_mapping_mcore_to_hf` | MCore→HF | `"decoder.layers.0.self_attention.linear_qkv.weight"` | `["model.layers.0.self_attn.q_proj.weight", ...]` |
| `_weight_to_mcore_format` | HF→MCore | 参数名 + HF 张量列表 | 融合后的单个 `torch.Tensor` |
| `_weight_to_hf_format` | MCore→HF | 参数名 + 融合张量 | 拆分后的多个 HF 张量 |
| `_build_config` | — | `hf_config` → | `TransformerConfig` |

### 2.4 两种权重同步模式

| 模式 | CLI 参数 | 转换入口 | 使用场景 |
|------|----------|---------|------|
| **raw**（**本次使用**） | `--megatron-to-hf-mode raw` | `convert_qwen3_5_to_hf()` → `_split_in_proj_weight()` | 训练中每 step 高频同步 |
| **bridge** | `--megatron-to-hf-mode bridge` | `Bridge.export_weights()` → `Bridge._weight_to_hf_format()` | 离线导出 |

**两者的核心区别：**
```
raw 模式 (训练实际使用):
  Megatron params → tp all_gather → convert_qwen3_5_to_hf(param, name, args)
    → 对 GDN fused in_proj: _split_in_proj_weight() → split_gdn_linear_weights()
    → 对 GDN conv1d:       _deinterleave_conv1d() → deinterleave_gdn_conv1d()

bridge 模式 (离线):
  Megatron params → Bridge.export_weights()
    → _weight_name_mapping_mcore_local_to_global (VPP/EP 映射)
    → _weight_merge_across_tp (TP gather)
    → _weight_to_hf_format (GDN 拆分)
```

**对 GDN 而言，两个模式使用同一套 `gdn_param_mapping` 工具函数**，算法正确性一致。raw 模式没有 Bridge 的映射解析和 EP 处理，高频调用更轻量。

## 3. Qwen3.6 模型结构

### 3.1 VLM 嵌套结构

Qwen3.6 是 VLM，文本权重在 `model.language_model.*` 下：

```
Qwen3_5Model (VLM)
├── vision_tower          — 视觉编码器（本次迁移不涉及）
├── language_model        — 映射用 model.language_model.* 前缀
│   ├── embed_tokens
│   ├── layers[0..N-1]    — DecoderLayer × N (N=40)
│   └── norm
└── lm_head               — 在 VLM 顶层
```

### 3.2 混合注意力层

每 interval=4 层用一个标准 Full Attention，其余用 GatedDeltaNet（linear attention）：

| 层索引 | 注意力类型 |
|--------|-----------|
| Layer 0, 1, 2 | linear_attn (GatedDeltaNet) |
| Layer 3 | self_attn (Full Attention) |
| Layer 4, 5, 6 | linear_attn (GatedDeltaNet) |
| Layer 7 | self_attn (Full Attention) |
| ... | ... (pattern 重复，约 3/4 的层是 GDN) |

### 3.3 GDN 特有的维度参数

| 参数 | 含义 |
|------|------|
| `linear_num_value_heads` | GDN 的 V head 数 |
| `linear_num_key_heads` | GDN 的 K head 数（类似 GQA） |
| `linear_key_head_dim` | GDN 的 K head 维度 |
| `linear_value_head_dim` | GDN 的 V head 维度 |
| `linear_conv_kernel_dim` | causal conv 的 kernel size |

## 4. Bridge 继承链：权重转换框架的完整面貌

继承链（从上到下）：

| 层级 | 类 | 所在文件 | 职责 |
|------|-----|---------|------|
| 基类 | `Bridge` | `mbridge/core/bridge.py` | 定义转换流程 |
| LLM 层 | `LLMBridge` | `mbridge/core/llm_bridge.py` | 添加 `_CONFIG_MAPPING` + `_build_base_config` |
| Qwen MoE 层 | `Qwen2MoEBridge` | `mbridge/models/qwen2moe.py` | Qwen MoE 映射: `model.*` → `model.language_model.*` |
| **本次迁移** | **`Qwen3_5Bridge`** | `vime_plugins/mbridge/qwen3_5.py` | GDN fused in_proj/conv1d + 3D expert tensor |

Qwen3_5Bridge 重写了 6 个方法：

| 重写点 | 父类行为 | 为什么改 |
|--------|---------|---------|
| `_ATTENTION_MAPPING` | 只有 linear_qkv (QKV 三合一) | 加上 GDN fused in_proj (4→1 融合) 和 fused conv1d |
| `_MLP_MAPPING` | 每 expert 存独立 tensor | MoE experts 用 fused 3D tensor `[num_experts, ...]` |
| `_CONFIG_MAPPING` | `ffn_hidden_size` → `intermediate_size` (必须) | MoE 模型没有 `intermediate_size`，改可选 |
| `_weight_to_mcore_format` | merge qkv → cat(q,k,v) | 加上 Route B 的 4→1 融合 + TP 交错 |
| `_weight_to_hf_format` | split qkv → q,k,v | 加上 Route B 的 1→4 拆分（bridge 模式用） |
| `_build_config` | 标准 Qwen2MoE | Qwen3.5 特有: qk_layernorm, attention_output_gate, moe_router_pre_softmax=False |

## 5. 迁移方案：为什么选 Route B（fused in_proj）

### 5.1 两种方案对比

| 方面 | Route A: 保持 4 路独立 | Route B: fused in_proj (采用) |
|------|----------------------|------------------------------|
| **in_proj 实现** | 4 个 `ColumnParallelLinear` | 1 个 `ColumnParallelLinear` |
| **TP 切分** | 每个独立切 → Q/K/V/Z/B/A 的 head 对齐复杂 | 合并后按 `[q₀,k₀,v₀,z₀,b₀,a₀, q₁,...]` 统一交错切 |
| **与 Megatron 原生对齐** | 需大幅魔改 TransformerBlock | 与 Megatron `gated_delta_net.py` 结构一致 |
| **dist-ckpt 兼容** | 4 个 ShardedTensor → 跨 TP resharding 复杂 | 1 个 fused + `_split_tensor_factory` 拆 6 段 |
| **kernel launch** | 4 次 GEMM | 1 次 GEMM + 1 次 split |

**选择 Route B 的根本原因：TP 兼容性。** 从 `num_v_heads`/`num_k_heads` 切分的角度看，fused 方案通过预交错排列保证每个 rank 上的 6 段精确对齐。

## 6. 逐文件详解

按 11 个文件讲解，涵盖权重转换工具、Bridge 映射、Kernel 后端、模型定义、CP 工具、转换脚本、raw 模式转换器、运行时权重更新、参数注册。每文件分三块：**已有框架 → 改动内容 → 为什么这样改**。

### 📄 文件 1: `vime_plugins/mbridge/gdn_param_mapping.py` (新增, 282 行)

**角色：** GDN 权重转换的**纯 PyTorch 工具库**——6 个函数，零外部依赖。是整个迁移的「单一可信源」。

**被谁调用？** 三个地方：`mbridge/qwen3_5.py`（bridge 模式）、`megatron_to_hf/qwen3_5.py`（raw 模式）、自身。

| 函数 | 方向 | 输入 → 输出 |
|------|------|------------|
| `_fuse_gdn_separate_to_grouped` | HF→中间 | `(qkv,z,b,a)` 4 flat → `(qkvz,ba)` head-grouped |
| `merge_gdn_linear_weights` | 中间→MCore | `(qkvz,ba)` + tp_size → fused in_proj（TP 交错） |
| `interleave_gdn_conv1d` | HF→MCore | `conv[conv_dim,1,k]` → TP 交错全局 |
| `split_gdn_linear_weights` | MCore→中间 | fused in_proj → `(qkvz,ba)`（merge 的逆） |
| `_split_gdn_grouped_to_separate` | 中间→HF | `(qkvz,ba)` → `(qkv,z,b,a)`（fuse 的逆） |
| `deinterleave_gdn_conv1d` | MCore→HF | TP 交错全局 → `conv[conv_dim,1,k]` |

**核心算法** `merge_gdn_linear_weights`：
1. qkvz 和 ba 各 reshape 成 `[num_qk_heads, per_group_dim, hidden_size]`
2. 拆出 q/k/v/z/b/a 共 6 个段
3. 每段 reshape 成 `[tp_size, local, hidden_size]`
4. `torch.cat([q,k,v,z,b,a], dim=1)` — 顺序 `[q₀,k₀,v₀,z₀,b₀,a₀, q₁,...]` 是**硬约定**

### 📄 文件 2: `vime_plugins/mbridge/qwen3_5.py` (修改)

**角色：** Qwen3.6 的 Bridge 实现。改前是能工作的 Qwen3.5 Bridge，但 GDN 层走直通映射（1→1），无 TP 支持。

**改动 A — 导入 GDN helper (L19-26)：** 从 `gdn_param_mapping` import 6 个函数。

**改动 B — `_ATTENTION_MAPPING` (L63-72)：** 新增 fused in_proj（1→4）和 fused conv1d（1→1 但需 TP 交错）映射。从直通映射移除 5 个已进入显式映射的参数。

**改动 C — `_weight_to_mcore_format` (L334-342)：** 在父类的 linear_qkv merge 外增加 GDN 融合。调用 `_gdn_fuse_separate_to_grouped` + `_gdn_merge_linear_weights`，返回全局张量让 Bridge 自动 TP chunk。

**改动 D — `_weight_to_hf_format` (L349-361)：** bridge 模式的 MCore→HF 转换，训练实际用 raw 模式但 bridge 模式也须实现以保持完整。

**改动 E — `_build_config` (L363-405)：** Qwen3.5 特有配置：`moe_router_pre_softmax=False`、`qk_layernorm=True`、`attention_output_gate=True`。MoE 模型的 `ffn_hidden_size` 映射改为可选（`(..., None)`）。

### 📄 文件 3: `vime_plugins/models/qwen_gdn_backend.py` (新增, 86 行)

**角色：** GDN 核心算子的后端分发器。

**`get_chunk_gated_delta_rule` — 三种后端：**
```python
if backend == "fla":      from fla.ops.gated_delta_rule import ...     # NVIDIA GPU
if backend == "flashqla": from flash_qla import chunk_gated_delta_rule  # NVIDIA SM90+
if backend == "npu":      from mindspeed.ops.chunk_gated_delta_rule...  # Ascend NPU ← 新增
```

**`get_causal_conv1d` — 新增，仅 NPU 有效：** MindSpeed Triton kernel，可通过 `QWEN36_CAUSAL_CONV1D_IMPL=eager` 回退到 `F.silu(F.conv1d(...))`。

### 📄 文件 4: `vime_plugins/models/qwen3_5.py` (重大修改, 629 行)

**角色：** Qwen3.6 在 Megatron 中的模型定义——改动最大。旧版使用 4 个独立 `nn.Linear` 和 fla `ShortConvolution`，无 TP/CP/SP 支持。

**新增 `_mark_tp`：** 标记非 Megatron 层参数为 TP 分片（Conv1d.weight、dt_bias、A_log 等）。

**新增 `Qwen3_5MoeRMSNormGated`：** NPU 用 torch 实现的 Gated RMSNorm（`rms_norm(x) * silu(gate)`），替代 fla 的 CUDA kernel。

**核心重写 `Qwen3_5GatedDeltaNet`：**

| 组件 | 旧版 | 新版 (Route B) |
|------|------|---------------|
| in_proj | 4 个 `nn.Linear` | 1 个 `ColumnParallelLinear(h, in_proj_dim)` |
| conv1d | `fla.ShortConvolution` | `nn.Conv1d(conv_dim//tp)` + Triton/eager dispatch |
| dt_bias / A_log | 全局 `nv` | `nv//tp` + `_mark_tp(dim=0)` |
| norm | 无条件 `FusedRMSNormGated` | 后端感知: fla 或 `Qwen3_5MoeRMSNormGated` |
| out_proj | `nn.Linear` | `RowParallelLinear(v_dim, h)` |

**forward 数据流：** `ColumnParallelLinear` → split [qkv,z,b,a] → `_conv(conv1d, qkv)` [双后端] → `chunk_gated_delta_rule` → `norm(out, z)` [后端感知] → `RowParallelLinear` [自动 TP all-reduce]

**`sharded_state_dict`：** `_split_tensor_factory` 把 in_proj.weight 再拆成 6 个独立 ShardedTensor，保证 cross TP-size resharding 正确。

**`get_qwen3_5_spec`：** 在 Megatron TransformerLayer 中把 `linear_attention` 层的 self_attention 替换为含 GDN 的 Attention wrapper。

### 📄 文件 5: `vime_plugins/models/hf_attention.py` (修改)

**角色：** HF 风格的 Attention wrapper 基类。增加 CP mode 判断：

```python
cp_mode = os.environ.get("QWEN36_CP_MODE", "ulysses")
if cp_size > 1 and cp_mode == "gather_dup":
    # gather_dup: 全序列 gather + GDN 冗余计算（旧基线）
    # ulysses (默认): 保持 CP-scattered，由 GDN._forward_cp 处理（省内存）
```

### 📄 文件 6: `vime_plugins/models/gdn_cp_utils.py` (新增, 258 行)

**角色：** Context Parallel 工具函数。新版 Megatron 有 `tensor_a2a_cp2hp` 等 CP helper，但 vime 用的老版 Megatron 没有。在老版底层 primitive 之上按新版接口签名实现。

**Part A — packed×CP THD 索引重排 (纯 torch)：**

| 函数 | 作用 |
|------|------|
| `thd_get_partitioned_indices_torch()` | 返回 cp_rank 在 packed buffer 中的 token 下标 |
| `undo_attention_load_balancing_thd()` | zigzag 负载均衡 → 自然顺序 |
| `redo_attention_load_balancing_thd()` | 自然顺序 → zigzag 负载均衡（undo 逆） |
| `compute_cp_seqlen_padding()` | 每条序列 pad 到 2*cp 整数倍 |
| `pad_packed_for_cp()` | 真实 token 散入 padded buffer |
| `unpad_packed_for_cp()` | 从 padded buffer 取回真实 token |

Part A 为什么需要：vime 用 packed sequence（不等长序列拼接），老版 Megatron 的 undo/redo 只支持非 packed（整段 zigzag）。

**Part B — 高层 CP helper（签名与新 Megatron `gated_delta_net.py` 对齐）：**

| 函数 | 签名 | 作用 |
|------|------|------|
| `get_parameter_local_cp` | `(param, dim, cp_group, split_sections)` | 取本 CP rank 的本地参数切片 |
| `tensor_a2a_cp2hp` | `(tensor, seq_dim, head_dim, cp_group)` | CP→HP all-to-all（底层走老栈 `_all_to_all_cp2hp`） |
| `tensor_a2a_hp2cp` | `(tensor, seq_dim, head_dim, cp_group)` | HP→CP all-to-all（底层走老栈 `_all_to_all_hp2cp`） |

### 📄 文件 7: `convert_qwen36.sh` + `tools/convert_hf_to_torch_dist.py` (新增/修改)

**角色：** HF → Megatron torch_dist 的权重转换。

**`convert_qwen36.sh`（新增）：** 8 GPU torchrun，`--qwen-gdn-backend npu` 触发 Route B 融合。

**`convert_hf_to_torch_dist.py` 三个 NPU 修复：**
1. **导入顺序**：GDN-patched megatron 在模块加载时 import mindspeed triton ops → 需要 `torch.npu` 已注册。先 `import torch_npu` 再 import megatron。
2. **hccl device_id**：`dist.init_process_group(backend="hccl")` 不能传 device_id（会导致 gloo sub-group 失败）。
3. **`--qwen-gdn-backend` 参数**：默认 fla 在 NPU 上不可用。

### 📄 文件 8: `scripts/run_qwen36_35b_a3b_dapo_math_npu.sh` (新增)

16 NPU 的 DAPO RL 训练脚本。关键参数：

```bash
--tensor-model-parallel-size 2     # TP=2, in_proj 沿 head 切 2 份
--sequence-parallel                # SP: LayerNorm 沿序列切
--expert-model-parallel-size 8     # EP=8: 256 experts 分 8 ranks
--qwen-gdn-backend npu             # MindSpeed AscendC GDN 算子
--vllm-gpu-memory-utilization 0.30 # 训练+推理共享 16 NPU
--vllm-enable-sleep-mode           # 训练时 vLLM 卸载到 host
--eps-clip 0.2 --eps-clip-high 0.28  # DAPO decoupled clip
```

### 📄 文件 9: `vime/backends/megatron_utils/megatron_to_hf/qwen3_5.py` (修改)

**角色：** raw 模式下的 MCore→HF 转换器——训练中**每 step 都会被调用的高频路径**。

改前不支持 GDN fused in_proj，所有 `linear_attn.in_proj.*` 和 `conv1d.*` 会走到 `raise ValueError`。

**新增 `_split_in_proj_weight(param, prefix, args)`：** 重用 `gdn_param_mapping` 的 `split_gdn_linear_weights` + `_split_gdn_grouped_to_separate`。

**新增 `_deinterleave_conv1d(param, prefix, args)`：** 调用 `deinterleave_gdn_conv1d` 做 TP 去交错。

**`convert_qwen3_5_to_hf` 新增 3 条路由：** `in_proj.weight` → `_split_in_proj_weight`、`conv1d.weight` → `_deinterleave_conv1d`、`A_log`/`norm.weight` → fp32 upcast。

A_log 和 norm.weight 的 fp32 upcast 是数值稳定性修复：fp16 精度不足会导致 GDN 递归计算误差。

### 📄 文件 10: 运行时权重更新管线（3 文件修改）

**A) `update_weight_from_tensor.py` — IPC 权重更新生命周期（新增方法）**

```python
class vLLMColocateWorkerExtension:
    def init_weight_transfer_engine(self, init_info):
        return None  # colocate IPC 路径不需要

    def start_weight_update(self, is_checkpoint_format=True):
        # layerwise_reload 撤销 vLLM post-load kernel fusion
        initialize_layerwise_reload(model)

    def finish_weight_update(self):
        # 恢复 post-load fusion
        finalize_layerwise_reload(model, self.model_config)
```

vLLM CUDA Worker 有这些方法，vllm-ascend NPUWorker 没有。搭配 `--no-offload-rollout` 使用（参数在 NPU 上，copy 是 d2d）。

**B) `update_weight_from_distributed.py` — 守护导入：** `try import HCCL` → fallback None，防止部分 NPU 构建崩溃。

**C) `vllm_engine.py` — sleep level 修复：** `release_memory_occupation(level=1)`。level=2 会丢弃权重对象、wake_up 后重建不带 `weight_loader` → RLHF weight update 报错。

### 📄 文件 11: `vime/utils/arguments.py` (修改, 1 行)

```python
# 修改前: choices=["fla", "flashqla"]
# 修改后:
choices=["fla", "flashqla", "npu"]
```

不加 `npu`，`--qwen-gdn-backend npu` 在参数解析阶段就被 argparse 拒绝。

## 7. 端到端数据流（含 raw 模式）

### Phase 1: 权重转换 (convert_qwen36.sh)

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | 读入 HF safetensors | `in_proj_qkv, in_proj_z, in_proj_b, in_proj_a, conv1d, ...` |
| 2 | `Bridge.load_weights()` → `_weight_to_mcore_format()` | 调用 `gdn_param_mapping` 的 `merge_gdn_linear_weights` + `interleave_gdn_conv1d` |
| 3 | 融合后 | fused `in_proj [in_proj_dim, h]`（6 段 TP 交错）+ conv1d（TP 交错全局） |
| 4 | TP chunk → 保存 | Megatron torch_dist / dist-ckpt（每 rank 存自己的 TP 分片） |

### Phase 2: RL 训练 (raw 模式权重同步)

每 step 循环执行以下阶段：

**2a — Actor 训练**（8 NPU, TP=2, EP=8）

| 组件 | 执行内容 |
|------|---------|
| GDN forward | `ColumnParallelLinear` → split [q,k,v,z,b,a] 6 段 → `_conv` (Triton/eager dispatch) → `chunk_gated_delta_rule` → `norm(out, z)` → `RowParallelLinear` |
| backward + step | `loss.backward()` → 梯度累积 → `optimizer.step()` |

**2b — Raw 模式权重同步**（Actor → Rollout）

| 步骤 | 操作 |
|------|------|
| 1 | 每 rank 对自己持有的 TP 分片做 `tp_all_gather` |
| 2 | `convert_qwen3_5_to_hf(param, name, args)`：对 `in_proj.weight` → `_split_in_proj_weight()` → `split_gdn_linear_weights()` → `_split_gdn_grouped_to_separate()` → 得 `[qkv, z, b, a]` 4 个 HF tensor；对 `conv1d.weight` → `_deinterleave_conv1d()` → 得 `[conv1d]` 1 个 HF tensor；A_log / norm.weight → upcast fp32 |
| 3 | `vLLMColocateWorkerExtension.start_weight_update()` → `initialize_layerwise_reload(model)` |
| 4 | IPC 传输 → `update_weights_chunk(update_info)` → `model.load_weights(weights)` |
| 5 | `finish_weight_update()` → `finalize_layerwise_reload(model)` |
| 6 | Rollout engine 继续 `generate()` |

**2c — Rollout + Reward**（8 NPU, vLLM）

vLLM 使用 HF 格式权重，生成新一批回答 → reward 模型打分 → 进入下一 step。

## 8. 总结

### 一句话总结

把 Qwen3.6 GDN 层从「4 个独立 `nn.Linear`」重构为「1 个 `ColumnParallelLinear` + TP/CP/SP 全支持」，并在两个权重同步路径（bridge 和 raw）都实现了 GDN 的融合/拆分。

### 硬约定（改动必须保持一致性！）

1. **fused in_proj 的 6 段顺序**：`[q|k|v|z|beta|alpha]` — `merge_gdn_linear_weights`、`forward` split、`sharded_state_dict`、`_split_in_proj_weight`、`_weight_to_hf_format` **这 5 处顺序必须完全一致**！
2. **conv1d TP 交错必须与 in_proj qkv 段对齐**
3. **GDN 内 `sequence_parallel=False`**（即使全局开了 SP）：卷积需连续上下文，递归需按时序递推
4. **A_log / norm.weight 导出时 upcast 到 fp32**：raw 和 bridge 模式都需要

### 完整文件清单（11 个文件）

| # | 文件 | 状态 | 作用 |
|---|------|------|------|
| 1 | `vime_plugins/mbridge/gdn_param_mapping.py` | **新增** | 6 个纯 torch 权重转换函数 |
| 2 | `vime_plugins/mbridge/qwen3_5.py` | **修改** | Bridge: fused in_proj/conv1d 映射 + 融合/拆分 |
| 3 | `vime_plugins/models/qwen_gdn_backend.py` | **新增** | Kernel dispatch: fla/flashqla/npu + conv1d 后端 |
| 4 | `vime_plugins/models/qwen3_5.py` | **重大修改** | 融合 GDN + TP/CP/SP + sharded_state_dict |
| 5 | `vime_plugins/models/hf_attention.py` | **修改** | CP mode: ulysses vs gather_dup |
| 6 | `vime_plugins/models/gdn_cp_utils.py` | **新增** | CP 工具: THD 重排 + CP helper |
| 7 | `convert_qwen36.sh` + `convert_hf_to_torch_dist.py` | **新增+修改** | HF→Megatron 转换 + NPU 适配 |
| 8 | `scripts/run_qwen36_35b_a3b_dapo_math_npu.sh` | **新增** | NPU RL 训练脚本 |
| 9 | `vime/backends/megatron_utils/megatron_to_hf/qwen3_5.py` | **修改** | raw 模式 GDN 拆分 |
| 10 | `update_weight_from_tensor.py` + `update_weight_from_distributed.py` + `vllm_engine.py` | **修改** | 运行时修复: layerwise reload, sleep level |
| 11 | `vime/utils/arguments.py` | **修改** | `--qwen-gdn-backend` choices 加入 `npu` |

---

*最后更新: 2026-06-23*